<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week5/lab2/WPI_week5_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation Quantification And Checkpoint Reuse

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

**Clinical disclaimer:** This lab uses public microscopy nuclei images for segmentation workflow practice. Physical measurements are simplified classroom calculations, not clinical measurements.


## Learning Objectives

By the end of this lab, you should be able to:

- Load a segmentation checkpoint created in another notebook or Colab session.
- Convert model probabilities into raw and post-processed masks.
- Compute area, simple volume estimates, and a severity-style index from masks.
- Explain how pixel spacing and slice thickness affect measurements.
- Describe why model portability matters for multi-lab workflows.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week5_lab2_responses_LastName_FirstName.docx`

Answer Q1-Q5 in the Word document, using 2-5 sentences per question.


## Workflow

This lab follows a post-segmentation analysis pipeline:

`Saved Lab 1 checkpoint -> Probability map -> Threshold -> Post-processing -> Mask -> Measurement -> Interpretation`

A different Colab session cannot see files from your previous session unless you downloaded them, saved them to Google Drive, or uploaded them again.


## Setup

Run this setup cell first. It installs small Python dependencies, clones the public course helper repo in Colab, and applies WPI plot styling.


In [ ]:
#@title Setup course environment
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scikit-image",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    repo_dir = Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])

sys.path.insert(0, str(repo_dir))

import shutil
import numpy as np
import matplotlib.pyplot as plt

from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, disk, remove_small_objects

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from wpi_ai_bootcamp.data import load_bbbc038_nuclei_segmentation_subset
from wpi_ai_bootcamp.notebook import make_wpi_overlay, setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)


## Data Loading

Use the same BBBC038 nuclei segmentation loader as Lab 1. The subset is deterministic so the checkpoint and examples stay aligned when you use the same parameters.


In [ ]:
images, masks, metadata = load_bbbc038_nuclei_segmentation_subset(
    max_samples=160,
    image_size=128,
    download=True,
    random_state=42,
)
source = metadata["source"]
print(source.name)
print(source.url)
print("images:", images.shape)
print("masks:", masks.shape)


## Hyperparameters

Only change values in this block when the notebook asks you to run a controlled comparison.


In [ ]:
# STUDENT-EDITABLE HYPERPARAMETERS
BASE_CHANNELS = 16
THRESHOLD = 0.5
MORPH_RADIUS = 2
MIN_OBJECT = 20
PIXEL_SPACING_UM = 0.65
SLICE_THICKNESS_UM = 1.0
SHOW_EXAMPLE_INDEX = 0
INTENSITY_SHIFT = 0.0
NUM_SLICES_FOR_3D = 5

# TODO: For Part 5, change exactly one value above and record the result in your Word response.


## Part 1 - Locate And Load The Lab 1 Checkpoint

Lab 2 can use the same Colab session, a Google Drive copy, or an uploaded checkpoint. If the file is missing, run the upload cell and select the `best_unet.pt` file from Lab 1.


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(),
        )
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        up = self.up(e2)
        return self.out(self.dec1(torch.cat([up, e1], dim=1)))


def dice_score_np(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    inter = (gt * pred).sum()
    return float((2 * inter + eps) / (gt.sum() + pred.sum() + eps))


def iou_score_np(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    inter = (gt * pred).sum()
    union = ((gt + pred) > 0).sum()
    return float((inter + eps) / (union + eps))


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = (probs * targets).sum(dim=dims)
    denom = probs.sum(dim=dims) + targets.sum(dim=dims)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def segmentation_loss(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits, targets) + dice_loss(logits, targets)


In [ ]:
LOCAL_CANDIDATES = [
    Path("best_unet.pt"),
    Path("week5_outputs/best_unet.pt"),
    Path("/content/drive/MyDrive/WPI_AI_Bootcamp/week5/best_unet.pt"),
]

checkpoint_path = next((path for path in LOCAL_CANDIDATES if path.exists()), None)

if checkpoint_path is None:
    print("No checkpoint found in the current session or default Drive path.")
    print("If you saved Lab 1 to Drive, mount Drive and the notebook will search again.")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        checkpoint_path = next((path for path in LOCAL_CANDIDATES if path.exists()), None)
    except Exception as exc:
        print("Drive mount skipped:", type(exc).__name__, str(exc)[:120])

if checkpoint_path is None:
    print("Upload the Lab 1 best_unet.pt file when prompted.")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            checkpoint_path = Path(next(iter(uploaded.keys())))
            print("Uploaded checkpoint:", checkpoint_path)
    except Exception as exc:
        print("Upload helper unavailable outside Colab:", type(exc).__name__)

if checkpoint_path is None or not checkpoint_path.exists():
    raise FileNotFoundError(
        "best_unet.pt was not found. Run Lab 1, download/save the checkpoint, then upload it or save it to Google Drive."
    )

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
base_channels = int(checkpoint.get("base_channels", BASE_CHANNELS)) if isinstance(checkpoint, dict) else BASE_CHANNELS
state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
model = TinyUNet(base_channels).to(DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("Loaded checkpoint:", checkpoint_path)
if isinstance(checkpoint, dict) and "validation_metrics" in checkpoint:
    print("Lab 1 validation metrics:", checkpoint["validation_metrics"])


### Part 1 Assessment - Checkpoint Reuse (20 pts)

Required notebook output: checkpoint path and successful model load.

Word response Q1: Why does a checkpoint need to be downloaded, uploaded, or saved to Drive when Lab 2 runs in a different Colab session?

Grading criteria: checkpoint is loaded and the response correctly explains Colab session storage.


## Part 2 - Generate Raw And Post-Processed Masks

Use the loaded model to create a probability map, threshold it, and apply morphology.


In [ ]:
class NucleiArrayDataset(Dataset):
    def __init__(self, images, masks, intensity_shift=0.0):
        shifted = np.clip(images + intensity_shift, 0.0, 1.0)
        self.images = torch.tensor(shifted, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]


def postprocess_mask(mask_np):
    mask_bool = mask_np.astype(bool)
    mask_bool = binary_closing(mask_bool, disk(MORPH_RADIUS))
    mask_bool = remove_small_objects(mask_bool, min_size=MIN_OBJECT)
    return mask_bool.astype(np.float32)


dataset = NucleiArrayDataset(images, masks, intensity_shift=INTENSITY_SHIFT)
image, gt_mask = dataset[SHOW_EXAMPLE_INDEX]
with torch.no_grad():
    logits = model(image.unsqueeze(0).to(DEVICE))
    probability = torch.sigmoid(logits).cpu().squeeze().numpy()
raw_mask = (probability > THRESHOLD).astype(np.float32)
processed_mask = postprocess_mask(raw_mask)
print("Probability range:", float(probability.min()), float(probability.max()))
print("Raw foreground fraction:", float(raw_mask.mean()))
print("Processed foreground fraction:", float(processed_mask.mean()))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()
axes[0].imshow(image.squeeze(), cmap="gray")
axes[0].set_title("Microscopy image")
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title("Target mask")
axes[2].imshow(probability, cmap="magma")
axes[2].set_title("Probability")
axes[3].imshow(raw_mask, cmap="gray")
axes[3].set_title("Raw mask")
axes[4].imshow(processed_mask, cmap="gray")
axes[4].set_title("Post-processed")
axes[5].imshow(make_wpi_overlay(image.squeeze().numpy(), processed_mask > 0.5))
axes[5].set_title("Processed overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


### Part 2 Assessment - Post-Processing (20 pts)

Required notebook output: probability map, raw mask, post-processed mask, and overlay.

Word response Q2: Why might post-processing be useful after thresholding a probability map?

Grading criteria: visual comparison is present and the response names a concrete post-processing effect.


## Part 3 - Quantification From Segmentation

Convert masks into simplified measurements. These calculations are for learning how segmentation affects downstream numbers.


In [ ]:
def area_um2(mask, pixel_spacing_um):
    return float(mask.sum() * pixel_spacing_um * pixel_spacing_um)


def volume_um3(mask_stack, pixel_spacing_um, slice_thickness_um):
    return float(mask_stack.sum() * pixel_spacing_um * pixel_spacing_um * slice_thickness_um)


def biomarker_index(mask):
    return float(mask.sum() / mask.size) if mask.size else 0.0

raw_area = area_um2(raw_mask, PIXEL_SPACING_UM)
processed_area = area_um2(processed_mask, PIXEL_SPACING_UM)
stack_processed = np.stack([processed_mask for _ in range(NUM_SLICES_FOR_3D)], axis=0)
processed_volume = volume_um3(stack_processed, PIXEL_SPACING_UM, SLICE_THICKNESS_UM)
processed_biomarker = biomarker_index(processed_mask)

print("Raw area (um^2):", round(raw_area, 2))
print("Processed area (um^2):", round(processed_area, 2))
print("Processed volume estimate (um^3):", round(processed_volume, 2))
print("Processed nuclei burden index:", round(processed_biomarker, 4))


In [ ]:
spacing_values = [0.4, 0.65, 1.0]
area_values = [area_um2(processed_mask, spacing) for spacing in spacing_values]

plt.figure(figsize=(6, 4))
plt.plot(spacing_values, area_values, marker="o", color=WPI_COLORS["crimson"])
plt.xlabel("Pixel spacing (um/pixel)")
plt.ylabel("Measured area (um^2)")
plt.title("Effect Of Pixel Spacing On Area")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 3 Assessment - Measurement (20 pts)

Required notebook output: area, volume, nuclei burden index, and pixel-spacing plot.

Word response Q3: Why is pixel count alone not enough to report physical area?

Grading criteria: measurements are produced and the response connects pixel spacing to physical units.


## Part 4 - 2D, 2.5D, And 3D Concepts

This is a conceptual comparison only. The model is still a 2D model trained on 2D microscopy images.


In [ ]:
area_2d = area_um2(processed_mask, PIXEL_SPACING_UM)
neighbor_masks = [processed_mask, processed_mask, processed_mask]
mask_25d = (np.mean(neighbor_masks, axis=0) > 0.5).astype(np.float32)
area_25d = area_um2(mask_25d, PIXEL_SPACING_UM)
volume_3d = volume_um3(stack_processed, PIXEL_SPACING_UM, SLICE_THICKNESS_UM)

plt.figure(figsize=(7, 4))
plt.bar(["2D area", "2.5D area", "3D volume"], [area_2d, area_25d, volume_3d], color=[WPI_COLORS["gray"], WPI_COLORS["accent_green"], WPI_COLORS["crimson"]])
plt.title("Conceptual 2D vs 2.5D vs 3D Comparison")
plt.tight_layout()
plt.show()

thickness_values = [0.5, 1.0, 2.0]
volume_values = [volume_um3(stack_processed, PIXEL_SPACING_UM, t) for t in thickness_values]
plt.figure(figsize=(6, 4))
plt.plot(thickness_values, volume_values, marker="o", color=WPI_COLORS["crimson"])
plt.xlabel("Slice thickness (um)")
plt.ylabel("Estimated volume (um^3)")
plt.title("Effect Of Slice Thickness On Volume")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 4 Assessment - Dimensional Thinking (20 pts)

Required notebook output: 2D/2.5D/3D comparison plot and slice-thickness plot.

Word response Q4: What is the key difference between 2D, 2.5D, and 3D segmentation concepts?

Grading criteria: plots are present and the response distinguishes slice-wise, contextual, and volumetric reasoning.


## Part 5 - One Controlled Comparison

Change exactly one parameter and compare the final measurement. Keep all other settings identical.

Recommended first comparison: change only `COMPARISON_PIXEL_SPACING_UM` below.


In [ ]:
# TODO: Change only this value for the required controlled comparison.
COMPARISON_PIXEL_SPACING_UM = 0.9

comparison_area = area_um2(processed_mask, COMPARISON_PIXEL_SPACING_UM)
comparison_volume = volume_um3(stack_processed, COMPARISON_PIXEL_SPACING_UM, SLICE_THICKNESS_UM)

print("Original pixel spacing:", PIXEL_SPACING_UM)
print("Comparison pixel spacing:", COMPARISON_PIXEL_SPACING_UM)
print("Original processed area:", round(processed_area, 2))
print("Comparison processed area:", round(comparison_area, 2))
print("Original processed volume:", round(processed_volume, 2))
print("Comparison processed volume:", round(comparison_volume, 2))


### Part 5 Assessment - Controlled Comparison (20 pts)

Required notebook output: original and comparison measurement values.

Word response Q5: Which single parameter did you change, and how did it affect area, volume, or biomarker interpretation?

Grading criteria: exactly one parameter changes, measurements are compared, and the interpretation is tied to the output.


## Optional Challenge

Try changing `INTENSITY_SHIFT` and rerun prediction. Explain whether the mask changes and why data shift can matter for biomedical segmentation models.


## Attribution

- Data: BBBC038 / 2018 Data Science Bowl nuclei segmentation dataset, downloaded at runtime from the Broad Bioimage Benchmark Collection.
- Dataset page: https://bbbc.broadinstitute.org/BBBC038
- Download used: `stage1_train.zip` from the BBBC038 page.
- Recommended citation: Caicedo, J. C., Goodman, A., Karhohs, K. W. et al. *Nucleus segmentation across imaging experiments: the 2018 Data Science Bowl*. Nature Methods 16, 1247-1253 (2019).
- License note: The BBBC038 page lists the image set copyright as CC0.
- Libraries: NumPy, Matplotlib, scikit-image, PyTorch, and course helper code.
